# ADT treatment intent -- localized/adjuvant vs. metastatic

Wrapper notebook. All logic lives in
`COMPASS/data_preprocessing/export_adt_intent_outputs.py`; this notebook loads
the cohort, calls `run()`, and displays what was written.

Every table and figure lands in one directory:

    <COMPASS_FIG_ROOT>/ADT_METASTATIC_FILTERING/

To regenerate outside Jupyter:

```
python COMPASS/data_preprocessing/export_adt_intent_outputs.py \
    --medications-path <MEDICATIONS.csv> \
    --patient-status-path <PT_INFO_STATUS.csv> \
    --icd-path <prostate_icd_data.csv>
```

The label's construction, its failure modes, and the reasoning behind each
cross-reference are documented in the module docstrings of
`classify_adt_intent.py` and `validate_adt_intent.py`.

## Configuration

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, ".")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl

from COMPASS.data_preprocessing.compile_COMPASS_cohort_data import load_patient_status
from COMPASS.data_preprocessing.adt_intent_trajectories import load_longitudinal
from COMPASS.data_preprocessing.export_adt_intent_outputs import (
    DEFAULT_STAGE_NOTE_LEVEL_PATH,
    resolve_out_dir,
    run,
)

DATA_ROOT = cp._PROFILE_OUTPUT_ROOT
MEDICATIONS_PATH = cp.PROFILE_SOURCES["MEDICATIONS"]
PATIENT_STATUS_PATH = cp.PROFILE_SOURCES["PT_INFO_STATUS_REGISTRATION"]

# Written by Stage 1 of 01_preprocessing.ipynb.
FLAGS_PATH = DATA_ROOT / "mrn_lists" / "icd_prostate_mrn_flags.csv"
ICD_PATH = DATA_ROOT / "prostate_icd_data.csv"

# Dated per-report stage, from PROFILE_data_processing/derive_cancer_annotations.ipynb.
STAGE_NOTE_LEVEL_PATH = DEFAULT_STAGE_NOTE_LEVEL_PATH

# Stage 2 output for the adt arm. Needed for the platinum/NEPC/AVPC KMs and the
# lab trajectories; the death KM does not depend on it.
LONGITUDINAL_ADT_PATH = DATA_ROOT / "longitudinal_prediction_data_adt.csv"

OUT_DIR = resolve_out_dir()

print(f"medications : {MEDICATIONS_PATH}")
print(f"pt status   : {PATIENT_STATUS_PATH}")
print(f"icd flags   : {FLAGS_PATH}  exists={FLAGS_PATH.exists()}")
print(f"icd record  : {ICD_PATH}  exists={ICD_PATH.exists()}")
print(f"stage notes : {STAGE_NOTE_LEVEL_PATH}  exists={Path(STAGE_NOTE_LEVEL_PATH).exists()}")
print(f"longitudinal: {LONGITUDINAL_ADT_PATH}  exists={LONGITUDINAL_ADT_PATH.exists()}")
print(f"output dir  : {OUT_DIR}")

## Load the ADT-exposed cohort

The label is built on every ADT-exposed patient, not just the modelled
eligible cohort -- `ELIGIBLE` is carried through as a column so downstream
code can narrow at use time.

In [ ]:
flags = pl.read_csv(FLAGS_PATH, infer_schema_length=0).with_columns(
    pl.col("DFCI_MRN").cast(pl.Float64, strict=False).cast(pl.Int64, strict=False),
    *[
        pl.col(c).cast(pl.Float64, strict=False).cast(pl.Int8, strict=False)
        for c in ("ADT_EXPOSED", "ELIGIBLE")
    ],
).drop_nulls("DFCI_MRN")

adt_exposed = flags.filter(pl.col("ADT_EXPOSED") == 1)
print(f"ICD-C61 patients      : {flags.height:,}")
print(f"  ADT_EXPOSED         : {flags['ADT_EXPOSED'].sum():,}")
print(f"  ELIGIBLE (modelled) : {flags['ELIGIBLE'].sum():,}")

status = load_patient_status(PATIENT_STATUS_PATH)
follow_up = status.select(
    pl.col("DFCI_MRN"),
    pl.col("DEATH_DATE").fill_null(pl.col("LAST_CONTACT_DATE")).alias("FOLLOW_UP_END_DATE"),
    pl.col("DEATH_DATE").is_not_null().cast(pl.Int64).alias("DEATH"),
).filter(pl.col("DFCI_MRN").is_in(adt_exposed["DFCI_MRN"]))

meds = cp.scan_source(MEDICATIONS_PATH).collect().with_columns(
    pl.col("DFCI_MRN").cast(pl.Float64, strict=False).cast(pl.Int64, strict=False)
).filter(pl.col("DFCI_MRN").is_in(adt_exposed["DFCI_MRN"]))

icds = pl.read_csv(ICD_PATH, infer_schema_length=0) if ICD_PATH.exists() else None

print(f"\nmedication rows : {meds.height:,}")
print(f"follow-up rows  : {follow_up.height:,}")
print(f"icd rows        : {icds.height:,}" if icds is not None else "icd rows        : [absent]")

# load_longitudinal cross-checks TREATMENT_ANCHOR_DATE against each patient's
# ADT_FIRST_DATE and refuses the arpi-arm file, whose anchor is first
# ARPI/taxane exposure -- same column names, silently wrong origin.
if LONGITUDINAL_ADT_PATH.exists():
    from COMPASS.data_preprocessing.export_adt_intent_outputs import build_labels
    longitudinal = load_longitudinal(
        LONGITUDINAL_ADT_PATH, labels=build_labels(meds, follow_up=follow_up)
    )
    print(f"longitudinal    : {longitudinal.height:,} rows, "
          f"{longitudinal['DFCI_MRN'].n_unique():,} patients")
else:
    longitudinal = None
    print(f"longitudinal    : [absent] run Stage 2 (cp.preprocess_labs) for the "
          f"adt arm to enable the\n                  platinum/NEPC/AVPC KMs and "
          f"lab trajectories")

## Build and write everything

`run()` classifies, joins on each cross-reference whose input is present,
writes every CSV and both figures, and reports what it skipped.

In [ ]:
labelled, out_dir = run(
    meds,
    follow_up=follow_up,
    icds=icds,
    stage_note_level_path=STAGE_NOTE_LEVEL_PATH,
    longitudinal=longitudinal,
)

## Review the outputs

In [ ]:
for p in sorted(out_dir.glob("*.csv")):
    print(f"{p.name:<36} {p.stat().st_size:>9,} bytes")
for p in sorted(out_dir.glob("*.png")):
    print(f"{p.name:<36} {p.stat().st_size:>9,} bytes")

### Class counts and the survival go/no-go

In [ ]:
print(pl.read_csv(out_dir / "summary_class_counts.csv"))

surv = out_dir / "summary_survival.csv"
if surv.exists():
    print()
    print(pl.read_csv(surv))
else:
    print("\n[skip] no survival table. report_survival needs a DEATH column;\n"
          "check that the patient-status frame carries one and that it\n"
          "survived the join in build_labels().")

### Stage and metastatic burden

Both are anchored on ADT start. `summary_stage_nearest` is the observation
closest to ADT start (windowed); `summary_stage_max` is the worst stage ever
recorded on each side (unwindowed), so their coverage differs by design.

In [ ]:
for name in ("summary_stage_nearest", "summary_stage_max",
             "summary_met_burden", "summary_met_site_pattern"):
    p = out_dir / f"{name}.csv"
    print(f"--- {name} ---")
    print(pl.read_csv(p) if p.exists() else "[skip] not written")
    print()

### Figures

In [ ]:
from IPython.display import Image, display

# Cross-reference panels, then time-to-event, then trajectories.
for name in ("stage_metburden", "max_stage",
             "km_death", "km_platinum", "km_nepc", "km_avpc",
             "lab_trajectories"):
    p = out_dir / f"{name}.png"
    if p.exists():
        print(name)
        display(Image(filename=str(p)))
    else:
        print(f"[skip] {name}.png not written")

### Time-to-event summaries

Event counts per class, printed alongside every KM: a curve built on a handful
of events looks just as confident as one built on hundreds, and only these
counts show the difference. Death covers all ADT-exposed patients; the other
three endpoints are defined only in the eligible survival cohort.

In [ ]:
for endpoint in ("death", "platinum", "nepc", "avpc"):
    for kind in ("km", "logrank"):
        p = out_dir / f"summary_{kind}_{endpoint}.csv"
        if p.exists():
            print(f"--- {kind} / {endpoint} ---")
            print(pl.read_csv(p))
            print()

### Known failure modes

ARPI-era drift and gap-threshold sensitivity. A label that swings sharply with
the gap threshold is measuring the threshold, not treatment intent.

In [ ]:
for name in ("summary_by_adt_start_year", "summary_gap_sensitivity"):
    p = out_dir / f"{name}.csv"
    print(f"--- {name} ---")
    print(pl.read_csv(p) if p.exists() else "[skip] not written")
    print()

## Using the label downstream

`adt_intent_labels.csv` is one row per ADT-exposed patient. Join on `DFCI_MRN`
and filter to `ADT_INTENT == "METASTATIC"` to drop completed adjuvant courses;
add `ELIGIBLE == 1` to narrow to the modelled cohort.

The cross-reference columns (`CANCER_STAGE`, `MAX_STAGE_BEFORE`,
`MAX_STAGE_AFTER`, `N_MET_SITES`, `MET_SITE_*`) are carried for auditing. They
are not independent of the label where both derive from the same ICD records,
so they should not be used as outcome variables against it.